# 02 — Cross-Lead Attention Fine-Tuning

Fine-tunes the Stage 2 LeadModel with `fusion_type='cross_attn'` using rectified ECG images and sparse COO pseudo-masks generated by notebook 01.


In [ ]:
!pip install --no-deps segmentation-models-pytorch==0.5.0
import os, sys, cv2, torch, pickle
import numpy as np
from timeit import default_timer as timer
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sys.path.insert(0, '/kaggle/input/datasets/zahouaniyacine/my-stage2-lead-model')
sys.path.append('/kaggle/input/datasets/takashisomeya/physionet-final-submission-models')
sys.path.append('/kaggle/input/datasets/hengck23/hengck23-demo-submit-physionet')

from stage2_lead_model import Net as LeadModel


In [ ]:
DEVICE = 'cuda'
FLOAT_TYPE = torch.float16
TARGET_TOTAL_EPOCHS = 4
EPOCHS_THIS_SESSION = 1
LR = 3e-5
BATCH_SIZE = 1
SAVE_EVERY_STEPS = 1000
RESUME_CKPT = '/kaggle/input/datasets/zahouaniyacine/attention-checkpoints-3/cross_attn_b6_last_full_2.pth'

SAVE_DIR = '/kaggle/working/checkpoints'
OUT_DIR = '/kaggle/working/outputs'
MASK_DIR = '/kaggle/input/notebooks/m1h4wk22/generate-pseudo-masks/output/masks'
RECT_DIR = '/kaggle/input/notebooks/m1h4wk22/generate-pseudo-masks/output/rectified'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

WINDOW_SIZE = 240
OFFSET = 416
xscale = 5000 / (2080 - 118)
IMGH, IMGW = int(1700), int(2200 * xscale + 1)
x0, x1 = 0, 5600
y0, y1 = 0, 1696
zero_mv = [703.5, 987.5, 1271.5, 1531.5]


In [ ]:
valid_id = sorted([f.replace('.mask-coo.npz', '') for f in os.listdir(MASK_DIR) if f.endswith('.mask-coo.npz')])
print('training samples:', len(valid_id))

def read_images(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMGW, IMGH), interpolation=cv2.INTER_LINEAR)
    image = image[y0:y1, x0:x1]
    H, W, _ = image.shape
    crops = []
    for zmv in zero_mv:
        h0, h1 = int(zmv - WINDOW_SIZE), int(zmv + WINDOW_SIZE)
        src_h0, src_h1 = max(0, h0), min(H, h1)
        dst_h0 = src_h0 - h0
        dst_h1 = dst_h0 + (src_h1 - src_h0)
        crop = np.zeros((WINDOW_SIZE * 2, W, 3), np.uint8)
        crop[dst_h0:dst_h1] = image[src_h0:src_h1]
        crops.append(crop)
    return np.stack(crops)

def load_sparse_mask(path):
    d = np.load(path)
    H, W = int(d['shape'][1]), int(d['shape'][2])
    mask = np.zeros((4, H, W), dtype=np.float32)
    for ch in range(4):
        if f'ch{ch}_y' in d and len(d[f'ch{ch}_y']) > 0:
            mask[ch, d[f'ch{ch}_y'], d[f'ch{ch}_x']] = d[f'ch{ch}_v']
    return mask


In [ ]:
class Stage2AttentionDataset(Dataset):
    def __init__(self, sample_ids):
        self.ids = sample_ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sample_id = self.ids[idx]
        lead_images = read_images(f'{RECT_DIR}/{sample_id}.rect.jpg')
        image_tensor = torch.from_numpy(lead_images.transpose(0, 3, 1, 2)).byte()
        mask = load_sparse_mask(f'{MASK_DIR}/{sample_id}.mask-coo.npz')
        mask_tensor = torch.from_numpy(mask).float().unsqueeze(1)
        return {'image': image_tensor, 'pixel': mask_tensor, 'sample_id': sample_id}


In [ ]:
def set_trainable(module, flag):
    for p in module.parameters():
        p.requires_grad = flag

def apply_progressive_unfreezing(model, global_epoch):
    set_trainable(model.encoder, False)
    set_trainable(model.decoder, True)
    set_trainable(model.fusion_modules, True)
    set_trainable(model.pixel_head, True)
    enc = model.encoder.model if hasattr(model.encoder, 'model') else model.encoder
    if global_epoch == 0:
        return 'encoder frozen'
    if hasattr(enc, 'blocks'):
        set_trainable(enc.blocks[-1], True)
        if global_epoch >= 2:
            set_trainable(enc.blocks[-2], True)
        if hasattr(enc, 'conv_head'):
            set_trainable(enc.conv_head, True)
        if hasattr(enc, 'bn2'):
            set_trainable(enc.bn2, True)
    return 'last 2 encoder blocks' if global_epoch >= 2 else 'last encoder block'


Final run result: the fourth epoch completed on 8,792 samples with average loss around `0.3697`; final weights reloaded with zero missing and zero unexpected keys.
